# FRED Chroma Query

In [4]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
from collections import Counter
from datetime import datetime, timezone


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.constants import FRED_COLLECTION_NAME, FRED_DB_NAME
from apps.agentic.core.document_loaders.fred_document_loader import FREDChromaDocumentLoader
from apps.agentic.core.utils import (set_chatgpt_env, set_langsmith_env, set_tavily_env, 
                                     set_github_env)

DB_PATH = str(Path("../../.db").resolve())

set_chatgpt_env(filedir="../../.keys")
set_langsmith_env(filedir="../../.keys")

## Verify Document Counts

In [3]:
FRED_DB_NAME, FRED_COLLECTION_NAME, DB_PATH, os.listdir(DB_PATH)

('fred',
 'fred',
 PosixPath('/Users/troy/Develop/gly.fish/yada/.db'),
 ['.DS_Store', 'research_library', 'fred', 'pdf_document_library', 'github'])

In [5]:
vs = FREDChromaDocumentLoader(DB_PATH).vectorstore
coll = vs._collection

In [6]:
client = vs._client
print([c.name for c in client.list_collections()])
print("Opened:", coll.name)
print("total docs:", coll.count())

['fred']
Opened: fred
total docs: 13028


## Verify Document Metadata

In [7]:
probe = coll.get(limit=5000)
metadata_list = probe.get("metadatas") if probe else []
metas = [m for m in metadata_list or [] if m]
len(metas)

5000

In [8]:
keys = set().union(*(m.keys() for m in metas)) if metas else set()
for key in sorted(keys):
    print(key)

category_id
category_name
category_path
filename
frequency
last_updated
leaf_name
observation_end
observation_start
popularity
seasonal_adjustment
series_id
series_title
units


## Search Examples 

In [15]:
vs = FREDChromaDocumentLoader(DB_PATH).vectorstore
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
    },
)

hits = retriever.invoke("Which Price series track Producer Price Indexes for energy commodities, and what are their observation ranges?")
[print(h) for h in hits]

page_content='# Producer Price Index by Commodity: Fuels and Related Products and Power

## Series Information
- **Series ID:** PPIENG
- **Frequency:** Monthly
- **Units:** Index 1982=100
- **Seasonal Adjustment:** Not Seasonally Adjusted
- **Observation Range:** 1926-01-01 – 2025-09-01
- **Last Updated:** 2025-11-25T09:50:16-06:00
- **Popularity:** 26

## Category Information
- **Category ID:** 33532
- **Category Name:** Fuels and Related Products and Power
- **Leaf Name:** Fuels and Related Products and Power
- **Category Path:** Prices > Producer Price Indexes (PPI) > Commodity Based > Fuels and Related Products and Power' metadata={'series_id': 'PPIENG', 'frequency': 'Monthly', 'filename': 'fred_prices_32455.yaml', 'seasonal_adjustment': 'Not Seasonally Adjusted', 'popularity': 26, 'observation_end': '2025-09-01', 'category_name': 'Fuels and Related Products and Power', 'leaf_name': 'Fuels and Related Products and Power', 'series_title': 'Producer Price Index by Commodity: Fuels an

[None, None, None, None, None, None, None, None]

## Metadata Queries

In [16]:
category_counts = Counter(m.get("category_name") for m in metas if "category_name" in m and m.get("category_name"))
for category, count in category_counts.most_common(20):
    print(f"{count:5d}  {category}")

 1000  Commodities
  726  House Price Indexes
  417  Employment Cost Index
  306  Intermediate Demand By Production Flow
  305  Chemicals and Allied Products
  255  Farm Products
  225  Inputs to Industries
  220  Final Demand
  207  Fuels and Related Products and Power
  202  Furniture and Household Durables
  176  Special Indexes
  114  Intermediate Demand By Commodity Type
   91  Health Care Services
   74  Food and Beverages
   57  Health Care Indexes
   51  Housing
   50  Hides, Skins, Leather, and Related Products
   47  Advertising Space and Time Sales
   45  Transportation
   45  Commodity and Services Groups


### Series

In [18]:
probe = coll.get(
    where={"category_name": "Housing"},
    limit=200
)
metadata_list = probe.get("metadatas") if probe else []
names = [m["series_title"] for m in metadata_list or [] if m]

for name in names:
    print(name)

Consumer Price Index for All Urban Consumers: Housing in U.S. City Average
Consumer Price Index for All Urban Consumers: Housing in U.S. City Average
Consumer Price Index for All Urban Consumers: Shelter in U.S. City Average
Consumer Price Index for All Urban Consumers: Fuels and Utilities in U.S. City Average
Consumer Price Index for All Urban Consumers: Household Energy in U.S. City Average
Consumer Price Index for All Urban Consumers: Household Furnishings and Operations in U.S. City Average
Consumer Price Index for All Urban Consumers: Rent of Primary Residence in U.S. City Average
Consumer Price Index for All Urban Consumers: Lodging Away from Home in U.S. City Average
Consumer Price Index for All Urban Consumers: Owners' Equivalent Rent of Residences in U.S. City Average
Consumer Price Index for All Urban Consumers: Owners' Equivalent Rent of Primary Residence in U.S. City Average
Consumer Price Index for All Urban Consumers: Fuel Oil and Other Fuels in U.S. City Average
Consumer

In [21]:
paths = [m["category_path"] for m in metadata_list or [] if m]
for path in paths:
    print(path)

Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > Housing
Prices > Consumer Price Indexes (CPI and PCE) > 